In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import danish

from shape_vs_intensity import config as C
from shape_vs_intensity import sim
from shape_vs_intensity.plotutils import use_style
use_style()

In [ ]:
# Set example Zernikes.  Arrays are Noll-indexed, so index 0 is unused.
z_example = np.zeros(C.JMAX + 1)
z_example[4] = 3e-6  # positive defocus
z_example[5] = -3e-6  # negative astig
z_example[7] = 2e-6  # positive coma
z_example[11] = -0.22e-6  # negative spherical
z_example[17] = -0.1e-6  # negative 2nd coma

In [ ]:
def render_donut(defocus, surface_brightness):
    z_ref = sim.make_reference(defocus=defocus)
    z_ref[: len(z_example)] += z_example

    model = danish.SingleDonutModel(
        sim.make_factory(surface_brightness=surface_brightness),
        z_ref=z_ref,
        z_terms=[],
        thx=0.0,
        thy=0.0,
        bkg_order=-1,
        seed=1,
        npix=C.NPIX,
    )
    base = model.model(1.0, 0.0, 0.0, C.FWHM, [], sky_level=None)
    flux = C.FLUX / base.sum()
    return model.model(flux, 0.0, 0.0, C.FWHM, [], sky_level=None)

# Repo convention: extra is the +defocalOffset / +Z4 side, intra is -Z4.
# With the +3 um Z4 perturbation above, intra is smaller and extra larger.
# Model only shapes
intra0 = render_donut(sim.signed_defocus(defocal_type="intra"), surface_brightness=False)
extra0 = render_donut(sim.signed_defocus(defocal_type="extra"), surface_brightness=False)

# Model with surface brightness fluctuations
intra1 = render_donut(sim.signed_defocus(defocal_type="intra"), surface_brightness=True)
extra1 = render_donut(sim.signed_defocus(defocal_type="extra"), surface_brightness=True)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.7), constrained_layout=True, dpi=150)

axes[0, 0].imshow(intra0, origin="lower")
axes[0, 1].imshow(extra0, origin="lower")
axes[1, 0].imshow(intra1, origin="lower")
axes[1, 1].imshow(extra1, origin="lower")

for ax in axes.flatten():
    ax.set(xticks=[], yticks=[])
axes[0, 0].set_title("Intrafocal")
axes[0, 1].set_title("Extrafocal")

fig.savefig(C.FIGDIR / "by-eye-example.pdf")